# RE_301 · Quant val aquest cotxe? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes per a una empresa que importa **cotxes de col·lecció dels anys 80**. Cada cop que arriba
una oferta, algú ha de mirar-se la fitxa tècnica i dir un preu. El cap vol una estimació
automàtica.

**La pregunta:** amb quin error, en dòlars, es pot estimar el preu d'un cotxe a partir de la seva fitxa? I aquest
número, és un número en què es pugui confiar?

L'enunciat complet, amb el diccionari de columnes, és a [RE_301](../RE_301_cotxes_preu.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
# El fitxer no porta capçalera: els noms són els del fitxer de descripció de la UCI
NOMS = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration', 'num-of-doors', 'body-style', 'drive-wheels', 'engine-location', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']

df = pd.read_csv(URL, header=None, names=NOMS)
print(df.shape)
df.head()

In [ ]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

## 1.1 · Què diu pandas, i què diu la UCI

Mira la forma i el tipus de cada columna. Compara-ho amb el fitxer de descripció de la UCI:
hi ha columnes que la UCI diu que són `continuous` i pandas no les llegeix com a números? Quines?

In [ ]:
text = df.select_dtypes(exclude="number").columns
print(len(text), list(text))

In [ ]:
# Les que la UCI declara "continuous" i pandas llegeix com a text
CONTINUES = ["normalized-losses", "wheel-base", "length", "width", "height", "curb-weight",
             "engine-size", "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
             "city-mpg", "highway-mpg", "price"]
sospitoses = [c for c in CONTINUES if c in text]
assert sospitoses == ["normalized-losses", "bore", "stroke", "horsepower", "peak-rpm", "price"]
sospitoses

Sis columnes numèriques es llegeixen com a text: `normalized-losses`, `bore`, `stroke`,
`horsepower`, `peak-rpm` i **`price`**, l'objectiu. Pandas diu **0 nuls**.

## 1.2 · Per què passa

Troba **quins valors** impedeixen que aquestes columnes siguin numèriques. Quan ho sàpigues,
**torna a carregar** el fitxer de manera que pandas ho entengui a la primera, sense arreglar les
columnes a mà una per una.

In [ ]:
estranys = {}
for col in sospitoses:
    no_num = df.loc[pd.to_numeric(df[col], errors="coerce").isna(), col]
    estranys[col] = no_num.value_counts().to_dict()
estranys

In [ ]:
n_estranys = sum(sum(d.values()) for d in estranys.values())
print("valors '?' a les columnes sospitoses:", n_estranys, "| a tota la taula:", (df == "?").sum().sum())

df = pd.read_csv(URL, header=None, names=NOMS, na_values="?")
assert df.select_dtypes("number").shape[1] == 16
assert df.isna().sum().sum() == (n_estranys + 2) == 59   # + 2 de num-of-doors, que és text
df.isna().sum().sum()

El nul està **disfressat de `?`**. N'hi ha 57 a les sis columnes numèriques i 2 més a
`num-of-doors` (que és text i per això no sortia a la llista): 59 en total, exactament els `NaN`
que apareixen en carregar amb `na_values="?"`. El fitxer `.names` ho diu (*Missing Attribute Values:
denoted by "?"*): qui l'ha llegit ho troba sense codi.

## 1.3 · L'inventari de forats

Ara que pandas els veu: quines columnes tenen valors que falten, quants, i quina proporció
de la taula són? Hi ha files sense **preu**?

In [ ]:
forats = df.isna().sum()[lambda s: s > 0].sort_values(ascending=False)
assert forats.sum() == df.isna().sum().sum()
assert forats["normalized-losses"] == 41 and forats["price"] == 4
pd.DataFrame({"nuls": forats, "proporcio": (forats / len(df)).round(3)})

`normalized-losses` té 41 forats (20 % de les files): és la decisió gran. Imputar-la amb la
mediana és defensable (és una dada de l'assegurança, no del cotxe); treure-la també. **Quatre
cotxes no tenen preu.**

## 2.1 · L'objectiu i les dades d'entrada

Separa `X` (la fitxa) i `y` (el preu). Què fas amb els cotxes que no tenen preu? Escriu per
què. (Pensa què vol dir *imputar* l'objectiu: amb quina resposta estaries entrenant el model?)

In [ ]:
dades = df.dropna(subset=["price"])
X = dades.drop(columns="price")
y = dades["price"]
assert y.isna().sum() == 0 and len(X) == len(y) == 201 and "price" not in X.columns
X.shape

Els 4 cotxes sense preu **es treuen**: imputar l'objectiu vol dir entrenar amb respostes
inventades (i avaluar-les!). Queden 201 cotxes.

## 2.2 · La referència

Abans de cap model de debò: quin error té un model que **sempre diu el preu mitjà**? Fes-ho
amb validació creuada de 5 trossos, i guarda la partició (`cv`) en una variable: la faràs servir
per a **tots** els models d'aquí endavant, perquè siguin comparables.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
mae_dummy = -cross_val_score(DummyRegressor(), X, y, cv=cv, scoring="neg_mean_absolute_error")
print(mae_dummy.round(0), mae_dummy.mean().round(0))

## 2.3 · El Pipeline

Munta un `Pipeline` que faci **tota** la preparació (omplir nuls de les numèriques, omplir nuls i
codificar les de text) i el model, i avalua'l amb la mateixa validació creuada. Prova com a mínim
dos models diferents.

Si et peta o et surten `nan`, llegeix l'avís sencer: et diu què ha vist en un tros de validació
que no havia vist en entrenar.

In [ ]:
num = make_column_selector(dtype_include="number")
txt = make_column_selector(dtype_exclude="number")

def prepara(handle_unknown="ignore"):
    return make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown=handle_unknown)), txt))

# Primer, l'error que es trobaran: sense handle_unknown, un tros de validació porta una marca
# que l'entrenament no ha vist
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fallits = cross_val_score(make_pipeline(prepara("error"), LinearRegression()), X, y, cv=cv,
                              scoring="neg_mean_absolute_error")
print("sense handle_unknown:", fallits)
print("marques amb un sol cotxe:", (X["make"].value_counts() == 1).sum())

In [ ]:
models = {
    "lineal": LinearRegression(),
    "arbre": DecisionTreeRegressor(random_state=0),
    "bosc": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
}
resultats = {"referència": mae_dummy}
for nom, m in models.items():
    s = -cross_val_score(make_pipeline(prepara(), m), X, y, cv=cv, scoring="neg_mean_absolute_error")
    assert np.isnan(s).sum() == 0
    resultats[nom] = s
taula = pd.DataFrame(resultats).agg(["mean", "std"]).T.round(0)
taula

In [ ]:
assert taula.loc["bosc", "mean"] < taula.loc["lineal", "mean"] < taula.loc["referència", "mean"]
assert 1300 < taula.loc["bosc", "mean"] < 1800

Amb `handle_unknown="error"` (el valor per defecte) alguns trossos donen `nan`: hi ha marques
amb un sol cotxe, i quan aquest cau a validació l'encoder no la coneix. Amb `"ignore"` funciona.
El bosc guanya clarament (MAE al voltant de 1.500 $), la lineal fa uns 1.900 $ i la referència
més de 5.500 $ (els números exactes són a la taula).

## 2.4 · Quin número dones al cap

Entrena el millor model amb totes les dades i calcula'n l'MAE **sobre les mateixes dades
d'entrenament**. Compara'l amb el de la validació creuada. Quin dels dos li diries al cap, i per
què?

In [ ]:
arbre = make_pipeline(prepara(), DecisionTreeRegressor(random_state=0)).fit(X, y)
bosc = make_pipeline(prepara(), RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)).fit(X, y)
comparacio = pd.DataFrame({
    "entrenament": [mean_absolute_error(y, arbre.predict(X)), mean_absolute_error(y, bosc.predict(X))],
    "validació creuada": [taula.loc["arbre", "mean"], taula.loc["bosc", "mean"]]},
    index=["arbre", "bosc"]).round(0)
assert (comparacio["entrenament"] < comparacio["validació creuada"]).all()
comparacio

L'arbre sense límit **memoritza**: el seu MAE d'entrenament és de 54 $ (només hi
ha l'error dels cotxes amb la mateixa fitxa i preus diferents). El bosc, uns pocs centenars. El
número per al cap és el de **validació creuada**, que es mesura sobre cotxes que el model no ha
vist.

## 2.5 · La resposta

Respon l'encàrrec: amb quin error es pot estimar el preu? Dona'l en dòlars i també relatiu al
preu típic d'un cotxe. Quant varia d'un tros de validació a un altre? Te'l creuries?

In [ ]:
mae, desv = taula.loc["bosc", "mean"], taula.loc["bosc", "std"]
print(f"MAE {mae:.0f} $ ± {desv:.0f} entre trossos")
print(f"relatiu a la mediana ({y.median():.0f} $): {mae / y.median():.1%}  · a la mitjana ({y.mean():.0f} $): {mae / y.mean():.1%}")

**Resposta:** el preu es pot estimar amb un error absolut mitjà d'uns **1.500 $** (validació
creuada de 5 trossos, bosc aleatori), aproximadament un 15 % de la mediana. Varia uns quants
centenars de dòlars d'un tros a l'altre: amb només 201 cotxes, el número és orientatiu. La
referència (dir sempre la mitjana) s'equivoca més de 5.500 $.

## 3.1 · Números escrits amb lletres

Hi ha columnes que són números però estan escrites amb paraules. Converteix-les a número i
compara l'error amb el que tenies quan les tractaves com a text. Millora? Per què podria ser?

In [ ]:
PARAULES = {"two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "eight": 8, "twelve": 12}
X2 = X.copy()
for col in ["num-of-doors", "num-of-cylinders"]:
    X2[col] = X2[col].map(PARAULES)
assert X2.select_dtypes("number").shape[1] == 17
s2 = -cross_val_score(make_pipeline(prepara(), RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)),
                      X2, y, cv=cv, scoring="neg_mean_absolute_error")
print(f"text: {taula.loc['bosc', 'mean']:.0f} $  ·  números: {s2.mean():.0f} $")

Amb el bosc la diferència és petita (dins del soroll entre trossos): un arbre pot tallar
igual una columna one-hot. On ajudaria més és en un model lineal, que amb el número aprofita
l'ordre.